# WIDEGAP 环境: QPO vs QCPO vs DQC-AC 对比实验

统一使用 `preset='widegap'` 环境 + MLP Actor `[8,8]`

In [2]:
import os, re, importlib
import torch
import numpy as np
import random
import wandb
from envs import RiskSensitiveEnv
from agents import QPO, QCPO, DQCAC

### 理论分位数


In [ ]:
# MC 模拟: 固定风险等级下的 mean vs Q_0.25 
import matplotlib.pyplot as plt

env_wg = RiskSensitiveEnv(max_steps=200, preset='widegap')
gamma = 0.99
n_episodes = 2000
risk_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]

means, quantiles = [], []
for r in risk_levels:
    returns = []
    for _ in range(n_episodes):
        state, _ = env_wg.reset()
        disc_return, disc_factor = 0, 1
        for _ in range(200):
            action = np.array([r], dtype=np.float32)
            state, reward, terminated, truncated, info = env_wg.step(action)
            disc_return += disc_factor * reward
            disc_factor *= gamma
            if terminated or truncated:
                break
        returns.append(disc_return)
    means.append(np.mean(returns))
    quantiles.append(np.percentile(returns, 25))
    print(f"r={r:.1f}: mean={means[-1]:.1f}, Q_0.25={quantiles[-1]:.1f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(means, quantiles, 'bo-', markersize=8)
for i, r in enumerate(risk_levels):
    ax.annotate(f'r={r}', (means[i]+0.5, quantiles[i]+0.3), fontsize=9)
ax.set_xlabel('Mean Return E[U]')
ax.set_ylabel('Q_0.25 Return')
ax.set_title('WIDEGAP: Risk-Return Frontier')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### QPO

In [ ]:
class QPO_Args:
    def __init__(self):
        self.env_name = 'RiskSensitiveEnv_widegap'
        self.log_interval = 100
        self.est_interval = 100
        self.q_alpha = 0.25
        self.max_episode = 2000000
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.algo_name = 'QPO'
        self.seed = 0

        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6

        self.actor_hidden_dims = [8, 8]
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.wandb_dir = None

args = QPO_Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

env = RiskSensitiveEnv(max_steps=200, preset='widegap')
agent_QPO = QPO(args, env)
agent_QPO.train()
wandb.finish()

### QCPO

In [ ]:
class QCPO_Args:
    def __init__(self):
        self.env_name = 'RiskSensitiveEnv_widegap'
        self.log_interval = 100
        self.est_interval = 100
        self.q_alpha = 0.25
        self.max_episode = 100000
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.algo_name = 'QCPO'
        self.seed = 0

        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6

        self.lambda_a = 0.01
        self.lambda_b = 5000
        self.lambda_c = 0.6
        self.outer_interval = 20

        self.density_estimate = True
        self.h_n = 0.05
        self.nu = 1
        self.quantile_threshold = 80

        self.actor_hidden_dims = [8, 8]
        self.normalize_gradient = 'scale'
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.wandb_dir = None

args = QCPO_Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

env = RiskSensitiveEnv(max_steps=200, preset='widegap')
agent_QCPO = QCPO(args, env)
agent_QCPO.train()
wandb.finish()

### DQC-AC

In [ ]:
class DQCAC_Args:
    def __init__(self):
        self.env_name = 'RiskSensitiveEnv_widegap'
        self.seed = 0
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self.max_episode = 100000
        self.log_interval = 100
        self.est_interval = 100
        self.gamma = 0.99

        self.q_alpha = 0.25
        self.quantile_threshold = 80
        self.outer_interval = 20

        self.init_std = float(np.sqrt(1e-1))
        self.theta_a = (10000 ** 0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9

        self.q_a = (10000 ** 0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6

        self.lambda_a = 0.01
        self.lambda_b = 5000
        self.lambda_c = 0.6

        # Critic
        self.emb_dim = [64, 64]
        self.critic_lr = 1e-3
        self.tau = 0.005
        self.target_update_interval = 100

        # Distributional TD
        self.num_quantiles = 32
        self.huber_kappa = 1.0
        self.density_bandwidth = 0.05

        # Replay Buffer
        self.buffer_capacity = 100000
        self.batch_size = 64
        self.min_buffer_size = 1000
        self.updates_per_episode = 10

        # Actor MLP
        self.actor_hidden_dims = [8, 8]
        self.algo_name = 'DQCAC'
        self.wandb_dir = None

args = DQCAC_Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

env = RiskSensitiveEnv(max_steps=200, preset='widegap')
agent_DQCAC = DQCAC(args, env)
agent_DQCAC.train()
wandb.finish()

### 评估

In [ ]:
from utils.evaluation import monte_carlo_evaluate

env = RiskSensitiveEnv(max_steps=200, preset='widegap')

results = {}
for name, agent in [('QPO', agent_QPO), ('QCPO', agent_QCPO), ('DQC-AC', agent_DQCAC)]:
    m, q = monte_carlo_evaluate(agent, env, num_episodes=1000)
    results[name] = {'mean': m, 'quantile': q}

print("=" * 60)
print("WIDEGAP 环境对比结果")
print("=" * 60)
for name, r in results.items():
    print(f"{name:10s} || mean={r['mean']:.2f}  Q_0.25={r['quantile']:.2f}")
print("-" * 60)
print(f"QCPO-QPO   || Δmean={results['QCPO']['mean']-results['QPO']['mean']:+.2f}")
print(f"DQC-QPO    || Δmean={results['DQC-AC']['mean']-results['QPO']['mean']:+.2f}")
print(f"threshold  || C=80")